In [36]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist, cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt 
import numpy as np

from tensorflow.keras import models, layers 

'''
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
'''

'\nfrom tensorflow.keras.models import Sequential\nfrom tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D\n'

In [45]:
# 1. feed-forward NN - Multiclass Image Classification on MNIST 

(x_train, y_train), (x_test, y_test) = mnist.load_data()
#plt.matshow(x_train[1]), x_train.shape --> (60000, 28, 28)

'''
norm_layer = layers.Normalization()
norm_layer.adapt(x_train)
'''
model = models.Sequential([
    layers.Input(shape=(28,28)), #ghost layer to map inputs, created by arg input_shape=(28, 28) of first layer automatically
    layers.Rescaling(1./255),
    layers.Flatten(),
    layers.Dense(units=128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(units=10, activation="softmax") #10 classes
])

# sparse_categorical_crossentropy loss since y_target variables are integers 
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=2, batch_size=32)
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"\n\ntest_loss : {test_loss}, test_accuracy : {test_acc}")
model.summary() # None --> bach size, each row for each sample 

Epoch 1/2
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9125 - loss: 0.2978
Epoch 2/2
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9581 - loss: 0.1425
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9695 - loss: 0.1033


test_loss : 0.10325192660093307, test_accuracy : 0.9695000052452087


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 28, 28)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305,312 (1.16 MB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 203,542 (795.09 KB)

In [35]:
y_pred_0 = np.argmax(model.predict(x_test[0:1])) # max probability arg/label # shape --> (1, 28,28), x_test[0] --> (28,28)
y_test_0 = y_test[0]
print(y_pred_0, y_test_0)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
7 7


In [47]:
# 2, CNN - Multiclass Image Classification on CIFAR10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train, y_train = x_train[:10000], y_train[:10000]
x_test, y_test = x_test[:2000], y_test[:2000]
y_test[0]

array([3], dtype=uint8)

In [48]:
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)
y_test[0] # to demonstrate using catgorical cross entopy loss for one hot encoded y target

array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])

In [53]:
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(32,32,3)), #batch will be another dimension while training
    layers.Conv2D(filters=32, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2,2), strides=(1,1), padding='same'),
    layers.Flatten(),
    layers.Dense(units=128, activation="relu"),
    layers.Dense(units=10, activation="softmax")
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=2, batch_size=64, validation_data=(x_test, y_test))
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"\n\ntest_loss : {test_loss}, test_accuracy : {test_acc}")
model.summary()

Epoch 1/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 45ms/step - accuracy: 0.2507 - loss: 2.2007 - val_accuracy: 0.3580 - val_loss: 1.7555
Epoch 2/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 44ms/step - accuracy: 0.4197 - loss: 1.5961 - val_accuracy: 0.4540 - val_loss: 1.5311
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4540 - loss: 1.5311


test_loss : 1.5310817956924438, test_accuracy : 0.45399999618530273


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_3 (Rescaling)         │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,589,856 (48.03 MB)

 Trainable params: 4,196,618 (16.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,393,238 (32.02 MB)

In [62]:
cifar10_classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

# Example usage with class index 3
print(cifar10_classes[np.argmax(model.predict(x_test[0:1]))])
print(cifar10_classes[int(y_test[0][0])])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
cat
airplane
